In [2]:
!pip install scipy

   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/36.6 MB ? eta -:--:--
    --------------------------------------- 0.5/36.6 MB 1.2 MB/s eta 0:00:32
    --------------------------------------- 0.8/36.6 MB 1.3 MB/s eta 0:00:28
   - -------------------------------------- 1.0/36.6 MB 1.5 MB/s eta 0:00:25
   - -------------------------------------- 1.6/36.6 MB 1.6 MB/s eta 0:00:22
   -- ------------------------------------- 1.8/36.6 MB 1.6 MB/s eta 0:00:22
   -- ------------------------------------- 2.4/36.6 MB 1.6 MB/s eta 0:00:21
   -- ------------------------------------- 2.6/36.6 MB 1.7 MB/s eta 0:00:21
   --- ------------------------------------ 2.9/36.6 MB 1.6 MB/s eta 0:00:21
   --- ------------------------------------ 2.9/36.6 MB 1.6 MB/s eta 0:00:21
   --- ------------------------------------ 2.9/36.6 MB 1.6 MB/s eta 0:00:21
   --- ------------------------------------ 3.1/36.6 MB 1.3 MB/s eta 0:00:27
   --- ------


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# PHASE 9 — AI VS DATA
# H1: Late deliveries are associated with lower review scores

import pandas as pd
import numpy as np
from scipy.stats import ttest_ind

# --------------------------------------------------
# 1. Load data
# --------------------------------------------------

orders = pd.read_csv("../data/olist_orders_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")


# --------------------------------------------------
# 2. Convert date columns
# --------------------------------------------------

date_columns = [
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")


# --------------------------------------------------
# 3. Calculate delivery delay
# --------------------------------------------------

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)


# --------------------------------------------------
# 4. Create late/on-time groups
# --------------------------------------------------

orders["delivery_status"] = np.where(
    orders["delivery_delay_days"] > 0,
    "Late",
    "On-time/Early"
)


# --------------------------------------------------
# 5. Join orders with reviews
# --------------------------------------------------

h1_data = orders[
    [
        "order_id",
        "delivery_delay_days",
        "delivery_status"
    ]
].merge(
    reviews[
        [
            "order_id",
            "review_score"
        ]
    ],
    on="order_id",
    how="inner"
)


# --------------------------------------------------
# 6. Remove missing values
# --------------------------------------------------

h1_data = h1_data.dropna(
    subset=[
        "delivery_delay_days",
        "review_score"
    ]
)


# --------------------------------------------------
# 7. Check dataset
# --------------------------------------------------

print("H1 dataset shape:", h1_data.shape)

print("\nDelivery status counts:")
print(h1_data["delivery_status"].value_counts())


# --------------------------------------------------
# 8. Compare average review scores
# --------------------------------------------------

average_scores = (
    h1_data
    .groupby("delivery_status")["review_score"]
    .agg(["mean", "count"])
    .round(2)
)

print("\nAverage review scores:")
print(average_scores)


# --------------------------------------------------
# 9. Calculate difference
# --------------------------------------------------

late_score = h1_data.loc[
    h1_data["delivery_status"] == "Late",
    "review_score"
]

on_time_score = h1_data.loc[
    h1_data["delivery_status"] == "On-time/Early",
    "review_score"
]

difference = (
    on_time_score.mean()
    - late_score.mean()
)

print("\nDifference in average review score:")
print(round(difference, 2))


# --------------------------------------------------
# 10. Statistical test — Independent t-test
# --------------------------------------------------

t_statistic, p_value = ttest_ind(
    late_score,
    on_time_score,
    equal_var=False
)

print("\nStatistical test results:")
print("T-statistic:", round(t_statistic, 4))
print("P-value:", p_value)


# --------------------------------------------------
# 11. Verdict
# --------------------------------------------------

alpha = 0.05

if p_value < alpha:
    verdict = "CONFIRMED"
else:
    verdict = "INCONCLUSIVE"

print("\nH1 VERDICT:", verdict)

H1 dataset shape: (96359, 4)

Delivery status counts:
delivery_status
On-time/Early    88658
Late              7701
Name: count, dtype: int64

Average review scores:
                 mean  count
delivery_status             
Late             2.57   7701
On-time/Early    4.29  88658

Difference in average review score:
1.73

Statistical test results:
T-statistic: -89.5508
P-value: 0.0

H1 VERDICT: CONFIRMED


In [4]:
# PHASE 9 — H2
# H2: Certain sellers have systematically poor customer experience

import pandas as pd
import numpy as np
from scipy.stats import ttest_ind

# --------------------------------------------------
# 1. Load data
# --------------------------------------------------

orders = pd.read_csv("../data/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")


# --------------------------------------------------
# 2. Connect orders -> sellers -> reviews
# --------------------------------------------------

seller_orders = order_items[
    ["order_id", "seller_id"]
].drop_duplicates()

h2_data = seller_orders.merge(
    reviews[
        ["order_id", "review_score"]
    ],
    on="order_id",
    how="inner"
)


# --------------------------------------------------
# 3. Calculate seller-level performance
# --------------------------------------------------

seller_summary = (
    h2_data
    .groupby("seller_id")
    .agg(
        average_review=("review_score", "mean"),
        reviewed_orders=("review_score", "count")
    )
    .reset_index()
)


# --------------------------------------------------
# 4. Keep sellers with enough reviewed orders
# --------------------------------------------------

seller_summary = seller_summary[
    seller_summary["reviewed_orders"] >= 20
].copy()


# --------------------------------------------------
# 5. Define poor-performing sellers
# --------------------------------------------------

poor_threshold = 3.5

seller_summary["performance_group"] = np.where(
    seller_summary["average_review"] < poor_threshold,
    "Poor-performing",
    "Other sellers"
)


# --------------------------------------------------
# 6. Display results
# --------------------------------------------------

print("Number of sellers analyzed:", len(seller_summary))

print("\nPerformance groups:")
print(
    seller_summary["performance_group"]
    .value_counts()
)

print("\nSeller review summary:")
print(
    seller_summary
    .groupby("performance_group")["average_review"]
    .agg(["mean", "count"])
    .round(2)
)


# --------------------------------------------------
# 7. Statistical test
# --------------------------------------------------

poor_sellers = seller_summary.loc[
    seller_summary["performance_group"] == "Poor-performing",
    "average_review"
]

other_sellers = seller_summary.loc[
    seller_summary["performance_group"] == "Other sellers",
    "average_review"
]

t_statistic, p_value = ttest_ind(
    poor_sellers,
    other_sellers,
    equal_var=False
)


# --------------------------------------------------
# 8. Statistical evidence
# --------------------------------------------------

print("\nStatistical test results:")
print("T-statistic:", round(t_statistic, 4))
print("P-value:", p_value)


# --------------------------------------------------
# 9. Verdict
# --------------------------------------------------

alpha = 0.05

if p_value < alpha:
    verdict = "CONFIRMED"
else:
    verdict = "INCONCLUSIVE"

print("\nH2 VERDICT:", verdict)

Number of sellers analyzed: 811

Performance groups:
performance_group
Other sellers      774
Poor-performing     37
Name: count, dtype: int64

Seller review summary:
                   mean  count
performance_group             
Other sellers      4.16    774
Poor-performing    3.19     37

Statistical test results:
T-statistic: -18.714
P-value: 5.715870098115911e-21

H2 VERDICT: CONFIRMED


In [5]:
# PHASE 9 — H3
# H3: Some product categories have unusually high complaint rates

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# --------------------------------------------------
# 1. Load data
# --------------------------------------------------

orders = pd.read_csv("../data/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")
products = pd.read_csv("../data/olist_products_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")
translation = pd.read_csv(
    "../data/product_category_name_translation.csv"
)


# --------------------------------------------------
# 2. Connect orders -> reviews -> products
# --------------------------------------------------

order_reviews = orders[
    ["order_id"]
].merge(
    reviews[
        ["order_id", "review_score"]
    ],
    on="order_id",
    how="inner"
)

order_items_products = order_items[
    ["order_id", "product_id"]
].merge(
    products[
        ["product_id", "product_category_name"]
    ],
    on="product_id",
    how="left"
)


# --------------------------------------------------
# 3. Add English category names
# --------------------------------------------------

order_items_products = order_items_products.merge(
    translation,
    on="product_category_name",
    how="left"
)

order_items_products["category"] = (
    order_items_products["product_category_name_english"]
    .fillna(order_items_products["product_category_name"])
)


# --------------------------------------------------
# 4. Avoid duplicate review counting
# --------------------------------------------------

category_orders = (
    order_items_products[
        ["order_id", "category"]
    ]
    .drop_duplicates()
    .merge(
        order_reviews,
        on="order_id",
        how="inner"
    )
)


# --------------------------------------------------
# 5. Create complaint flag
# --------------------------------------------------

category_orders["negative_review"] = np.where(
    category_orders["review_score"] <= 3,
    1,
    0
)


# --------------------------------------------------
# 6. Calculate category complaint rates
# --------------------------------------------------

category_summary = (
    category_orders
    .groupby("category")
    .agg(
        total_reviews=("review_score", "count"),
        negative_reviews=("negative_review", "sum")
    )
    .reset_index()
)

category_summary["complaint_rate"] = (
    category_summary["negative_reviews"]
    / category_summary["total_reviews"]
    * 100
)


# --------------------------------------------------
# 7. Keep categories with enough reviews
# --------------------------------------------------

category_summary = category_summary[
    category_summary["total_reviews"] >= 20
].copy()


# --------------------------------------------------
# 8. Display highest complaint categories
# --------------------------------------------------

top_complaints = category_summary.sort_values(
    "complaint_rate",
    ascending=False
)

print("Number of categories analyzed:", len(category_summary))

print("\nTop categories by complaint rate:")
print(
    top_complaints.head(10).round(2)
)


# --------------------------------------------------
# 9. Statistical test
# --------------------------------------------------

contingency_table = pd.crosstab(
    category_orders["category"],
    category_orders["negative_review"]
)

chi2, p_value, degrees_of_freedom, expected = chi2_contingency(
    contingency_table
)

print("\nChi-square test results:")
print("Chi-square statistic:", round(chi2, 4))
print("Degrees of freedom:", degrees_of_freedom)
print("P-value:", p_value)


# --------------------------------------------------
# 10. Verdict
# --------------------------------------------------

alpha = 0.05

if p_value < alpha:
    verdict = "CONFIRMED"
else:
    verdict = "INCONCLUSIVE"

print("\nH3 VERDICT:", verdict)

Number of categories analyzed: 67

Top categories by complaint rate:
                             category  total_reviews  negative_reviews  \
57                   office_furniture           1268               472   
27             fashio_female_clothing             41                15   
23                diapers_and_hygiene             27                 9   
30              fashion_male_clothing            111                36   
47                       home_confort            398               129   
4                               audio            348               110   
46                     home_comfort_2             23                 7   
41  furniture_mattress_and_upholstery             38                11   
19          construction_tools_safety            166                47   
58                     party_supplies             39                11   

    complaint_rate  
57           37.22  
27           36.59  
23           33.33  
30           32.43  
47         

In [6]:
# PHASE 9 — H4
# H4: High freight costs are associated with poor customer satisfaction

import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# --------------------------------------------------
# 1. Load data
# --------------------------------------------------

order_items = pd.read_csv("../data/olist_order_items_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")


# --------------------------------------------------
# 2. Calculate order-level price and freight
# --------------------------------------------------

order_costs = (
    order_items
    .groupby("order_id")
    .agg(
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum")
    )
    .reset_index()
)


# --------------------------------------------------
# 3. Calculate freight ratio
# --------------------------------------------------

order_costs["freight_ratio"] = np.where(
    order_costs["total_price"] > 0,
    order_costs["total_freight"] /
    order_costs["total_price"] * 100,
    np.nan
)


# --------------------------------------------------
# 4. Connect with customer reviews
# --------------------------------------------------

h4_data = order_costs.merge(
    reviews[
        ["order_id", "review_score"]
    ],
    on="order_id",
    how="inner"
)

h4_data = h4_data.dropna(
    subset=["freight_ratio", "review_score"]
)


# --------------------------------------------------
# 5. Display basic results
# --------------------------------------------------

print("H4 dataset shape:", h4_data.shape)

print("\nAverage review score:")
print(
    round(h4_data["review_score"].mean(), 2)
)

print("\nAverage freight ratio:")
print(
    round(h4_data["freight_ratio"].mean(), 2),
    "%"
)


# --------------------------------------------------
# 6. Statistical test
# --------------------------------------------------
# Spearman correlation is suitable because
# review_score is an ordinal rating.

correlation, p_value = spearmanr(
    h4_data["freight_ratio"],
    h4_data["review_score"]
)

print("\nSpearman correlation results:")
print("Correlation:", round(correlation, 4))
print("P-value:", p_value)


# --------------------------------------------------
# 7. Verdict
# --------------------------------------------------

alpha = 0.05

if p_value < alpha and correlation < 0:
    verdict = "CONFIRMED"
elif p_value >= alpha:
    verdict = "INCONCLUSIVE"
else:
    verdict = "REJECTED"

print("\nH4 VERDICT:", verdict)

H4 dataset shape: (98465, 5)

Average review score:
4.1

Average freight ratio:
30.86 %

Spearman correlation results:
Correlation: -0.026
P-value: 3.3549334970002794e-16

H4 VERDICT: CONFIRMED


In [7]:
# PHASE 9 — H5
# H5: Certain geographic regions have operational/customer-experience problems

import pandas as pd
from scipy.stats import kruskal

# --------------------------------------------------
# 1. Load data
# --------------------------------------------------

orders = pd.read_csv("../data/olist_orders_dataset.csv")
customers = pd.read_csv("../data/olist_customers_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")


# --------------------------------------------------
# 2. Connect orders -> customers -> reviews
# --------------------------------------------------

h5_data = (
    orders[["order_id", "customer_id"]]
    .merge(
        customers[
            ["customer_id", "customer_state"]
        ],
        on="customer_id",
        how="inner"
    )
    .merge(
        reviews[
            ["order_id", "review_score"]
        ],
        on="order_id",
        how="inner"
    )
)


# --------------------------------------------------
# 3. Remove missing values
# --------------------------------------------------

h5_data = h5_data.dropna(
    subset=["customer_state", "review_score"]
)


# --------------------------------------------------
# 4. Calculate state-level review performance
# --------------------------------------------------

state_summary = (
    h5_data
    .groupby("customer_state")
    .agg(
        average_review=("review_score", "mean"),
        reviewed_orders=("review_score", "count")
    )
    .reset_index()
)


# --------------------------------------------------
# 5. Keep states with enough reviews
# --------------------------------------------------

state_summary = state_summary[
    state_summary["reviewed_orders"] >= 20
].copy()


# --------------------------------------------------
# 6. Display lowest and highest states
# --------------------------------------------------

print("Number of states analyzed:", len(state_summary))

print("\nLowest-rated states:")
print(
    state_summary
    .sort_values("average_review")
    .head(10)
    .round(2)
)

print("\nHighest-rated states:")
print(
    state_summary
    .sort_values("average_review", ascending=False)
    .head(10)
    .round(2)
)


# --------------------------------------------------
# 7. Prepare groups for statistical test
# --------------------------------------------------

state_groups = []

for state in state_summary["customer_state"]:
    
    scores = h5_data.loc[
        h5_data["customer_state"] == state,
        "review_score"
    ]
    
    state_groups.append(scores)


# --------------------------------------------------
# 8. Kruskal-Wallis test
# --------------------------------------------------
# Tests whether review-score distributions
# differ across geographic regions.

statistic, p_value = kruskal(
    *state_groups
)


print("\nKruskal-Wallis test results:")
print("Statistic:", round(statistic, 4))
print("P-value:", p_value)


# --------------------------------------------------
# 9. Verdict
# --------------------------------------------------

alpha = 0.05

if p_value < alpha:
    verdict = "CONFIRMED"
else:
    verdict = "INCONCLUSIVE"

print("\nH5 VERDICT:", verdict)

Number of states analyzed: 27

Lowest-rated states:
   customer_state  average_review  reviewed_orders
21             RR            3.61               46
1              AL            3.75              414
9              MA            3.76              746
24             SE            3.81              349
13             PA            3.85              968
5              CE            3.85             1329
4              BA            3.86             3357
18             RJ            3.87            12765
16             PI            3.92              491
15             PE            4.01             1646

Highest-rated states:
   customer_state  average_review  reviewed_orders
3              AP            4.19               67
2              AM            4.18              147
17             PR            4.18             5038
25             SP            4.17            41690
10             MG            4.14            11625
22             RS            4.13             5483
11     